This code is designed to get a NIST csv file for a specie and parse it for e.g. SIESTA.

In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
class NISTParser():
    def __init__(self, filepath: str):
        self.filepath = filepath


    def toPandas(self,sep: str, header: int, print_head: bool) -> None:
        
        df = pd.read_csv(self.filepath, sep=sep, header=header)
        print(f"Dataframe loaded from {self.filepath.split('/')[-1]} has: {df.shape[0]} rows and {df.shape[1]} columns")
        print(f"Columns names are: {list(df.columns.values)}")
        
        if print_head:
            print(df.head(5))
        self.dataframe = df
    
    def selectColumns(self, columns: list) -> None:
        if not hasattr(self, 'dataframe'):
            raise ValueError("Dataframe not loaded. Please run toPandas() method first.")
        
        if not all(col in self.dataframe.columns for col in columns):
            missing_cols = [col for col in columns if col not in self.dataframe.columns]
            raise ValueError(f"The following columns are not in the dataframe: {missing_cols}")
        
        selected_df = self.dataframe[columns]
        self.dataframe = selected_df
        print(f"Selected columns: {columns}")


    def intensityFilter(self, threshold_up: float) -> None:
        if not hasattr(self, 'dataframe'):
            raise ValueError("Dataframe not loaded. Please run toPandas() method first.")
        
        if 'intens' not in self.dataframe.columns:
            raise ValueError("Column 'intens' not found in dataframe.")
        
        self.dataframe["intens"] = (self.dataframe["intens"].astype(str).str.replace(r"[^0-9]", "", regex=True).replace("", "0").astype(int))



        filtered_df = self.dataframe[self.dataframe['intens'] >= np.quantile(self.dataframe['intens'], threshold_up)]
        self.dataframe = filtered_df
        print(f"Filtered dataframe to keep intensities >= {threshold_up}. New shape: {self.dataframe.shape}")
        
    
    def exportDataframeNumpy(self, output_filepath: str) -> None:
        if not hasattr(self, 'dataframe'):
            raise ValueError("Dataframe not loaded. Please run toPandas() method first.")
        
        exported_array = self.dataframe.to_numpy()
        np.save(output_filepath, exported_array)
        print(f"Dataframe exported to numpy array at {output_filepath}")

        

In [3]:
test = pd.read_csv("./NIST_Atomic-Specie/NeI350-875nmtab.csv", sep='\t', header=0)

In [11]:
# test.head(20)


Neon = NISTParser("./NIST_Atomic-Specie/NeI350-875nmtab.csv")
Neon.toPandas(sep='\t', header=0, print_head=False)
Neon.intensityFilter(threshold_up=0.95)
Neon.selectColumns(['intens'])
# Neon.selectColumns(['obs_wl_air(nm)', 'intens'])
# Neon.exportDataframeNumpy("./NIST_Atomic-Specie/Neon0.98.npy")


Dataframe loaded from NeI350-875nmtab.csv has: 664 rows and 19 columns
Columns names are: ['obs_wl_air(nm)', 'unc_obs_wl', 'ritz_wl_air(nm)', 'unc_ritz_wl', 'intens', 'Aki(s^-1)', 'Acc', 'Ei(cm-1)', 'Ek(cm-1)', 'conf_i', 'term_i', 'J_i', 'conf_k', 'term_k', 'J_k', 'Type', 'tp_ref', 'line_ref', 'Unnamed: 18']
Filtered dataframe to keep intensities >= 0.95. New shape: (46, 19)
Selected columns: ['intens']


In [14]:
Neon.dataframe.describe()
# neon_array = Neon.dataframe.to_numpy()

# import matplotlib.pyplot as plt
# plt.hist(neon_array[:], bins=50)

,intens
count,46.000000
mean,26369.565217
std,24816.086804
min,10000.000000
25%,10000.000000
50%,15000.000000
75%,31250.000000
max,100000.000000
